# Camada Ouro Particionada — Processo Sicredi

Pipeline da Camada Ouro lendo do **star schema** (`workspace.gold.dim_*` + `workspace.gold.fact_transacoes_financeiras`) e gravando as tabelas analíticas no schema `workspace.gold`, usando **Spark SQL**:

- **`gold_transacoes_cartao_particionado`**: visão analítica via star join (fato + dim_cartao, dim_conta, dim_cliente, dim_estabelecimento). Particionada por `ano_mes`.
- **`gold_resumo_cliente_mes_particionado`**: indicadores mensais por cliente (quantidade, valor total, ticket médio). Particionada por `ano_mes`.
- **`gold_ranking_estabelecimento_particionado`**: ranking de estabelecimentos por valor total e quantidade de transações.
- **`gold_alertas_qualidade_particionado`**: resumo dos registros rejeitados nas quarentenas da Silver por entidade e tipo de erro.

Todas as tabelas Gold são reconstruídas via `CREATE OR REPLACE TABLE` (idempotente) a cada execução.

> **Arquitetura:** Silver → Star Schema (dim + fact) → Tabelas analíticas Gold. `gold_alertas_qualidade` lê das quarentenas Silver (dados de qualidade, não transacionais).

In [0]:
# ============================================================
# Parâmetros do pipeline Gold particionado
# Fonte: star schema (dim_* + fact_transacoes_financeiras)
# ============================================================
CATALOG = "workspace"
SILVER = f"{CATALOG}.silver"
GOLD   = f"{CATALOG}.gold"

print("Parâmetros configurados: star schema ->", GOLD, "| quarentenas ->", SILVER)

In [0]:
# ============================================================
# 1. GOLD_TRANSACOES_CARTAO_PARTICIONADO — visão analítica
# Base: star schema (fact_transacoes_financeiras + dimensoes)
# Star join: fato -> dim_cartao, dim_conta, dim_cliente, dim_estabelecimento
# Particionada por ano_mes.
# ============================================================

spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD}.gold_transacoes_cartao_particionado
USING DELTA
PARTITIONED BY (ano_mes)
AS SELECT
  f.id_transacao,
  f.data_transacao,
  f.ano_mes,
  f.valor,
  de.nome_estabelecimento,
  de.mcc,
  de.cidade_estabelecimento,
  dc3.id_cartao,
  dc3.bandeira,
  dc3.tipo_cartao,
  dc3.status_cartao,
  dc2.id_conta,
  dc2.tipo_conta,
  dc2.status_conta,
  dc.id_cliente,
  dc.nome   AS nome_cliente,
  dc.cidade AS cidade_cliente,
  dc.estado AS estado_cliente,
  CURRENT_TIMESTAMP() AS timestamp_processamento
FROM {GOLD}.fact_transacoes_financeiras f
INNER JOIN {GOLD}.dim_cartao          dc3 ON f.sk_cartao          = dc3.sk_cartao
INNER JOIN {GOLD}.dim_conta           dc2 ON f.sk_conta           = dc2.sk_conta
INNER JOIN {GOLD}.dim_cliente         dc  ON f.sk_cliente         = dc.sk_cliente
INNER JOIN {GOLD}.dim_estabelecimento  de  ON f.sk_estabelecimento = de.sk_estabelecimento
""")

qtd_gt = spark.table(f"{GOLD}.gold_transacoes_cartao_particionado").count()
print(f"GOLD_TRANSACOES_CARTAO: {qtd_gt} transações")
display(spark.sql(f"SELECT * FROM {GOLD}.gold_transacoes_cartao_particionado LIMIT 10"))

In [0]:
# ============================================================
# 2. GOLD_RESUMO_CLIENTE_MES_PARTICIONADO — indicadores mensais
# Agregação sobre gold_transacoes_cartao_particionado.
# Particionada por ano_mes.
# ============================================================

spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD}.gold_resumo_cliente_mes_particionado
USING DELTA
PARTITIONED BY (ano_mes)
AS SELECT
  id_cliente,
  nome_cliente,
  estado_cliente,
  ano_mes,
  COUNT(id_transacao)   AS qtd_transacoes,
  SUM(valor)             AS valor_total,
  ROUND(AVG(valor), 2)   AS ticket_medio,
  MIN(data_transacao)    AS primeira_transacao,
  MAX(data_transacao)    AS ultima_transacao,
  CURRENT_TIMESTAMP()    AS timestamp_processamento
FROM {GOLD}.gold_transacoes_cartao_particionado
GROUP BY id_cliente, nome_cliente, estado_cliente, ano_mes
""")

qtd_rcm = spark.table(f"{GOLD}.gold_resumo_cliente_mes_particionado").count()
print(f"GOLD_RESUMO_CLIENTE_MES: {qtd_rcm} combinações cliente x mês")
display(spark.sql(f"""
SELECT * FROM {GOLD}.gold_resumo_cliente_mes_particionado
ORDER BY id_cliente, ano_mes LIMIT 10
"""))

In [0]:
# ============================================================
# 3. GOLD_RANKING_ESTABELECIMENTO_PARTICIONADO — ranking
# por valor total e quantidade de transações (exclui estornadas).
# ============================================================

spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD}.gold_ranking_estabelecimento_particionado
USING DELTA
AS SELECT
  ROW_NUMBER() OVER (ORDER BY valor_total DESC, qtd_transacoes DESC)  AS ranking_valor,
  ROW_NUMBER() OVER (ORDER BY qtd_transacoes DESC, valor_total DESC) AS ranking_quantidade,
  nome_estabelecimento,
  mcc,
  qtd_transacoes,
  valor_total,
  ticket_medio,
  qtd_clientes_distintos,
  CURRENT_TIMESTAMP() AS timestamp_processamento
FROM (
  SELECT
    nome_estabelecimento,
    mcc,
    COUNT(id_transacao)        AS qtd_transacoes,
    SUM(valor)                  AS valor_total,
    ROUND(AVG(valor), 2)        AS ticket_medio,
    COUNT(DISTINCT id_cliente)  AS qtd_clientes_distintos
  FROM {GOLD}.gold_transacoes_cartao_particionado
  GROUP BY nome_estabelecimento, mcc
) agg
""")

qtd_re = spark.table(f"{GOLD}.gold_ranking_estabelecimento_particionado").count()
print(f"GOLD_RANKING_ESTABELECIMENTO: {qtd_re} estabelecimentos (par nome x mcc)")
display(spark.sql(f"""
SELECT * FROM {GOLD}.gold_ranking_estabelecimento_particionado
ORDER BY ranking_valor LIMIT 10
"""))

In [0]:
# ============================================================
# 4. GOLD_ALERTAS_QUALIDADE_PARTICIONADO — resumo das quarentenas
# O motivo_erro pode conter erros combinados ("A | B");
# SPLIT + EXPLODE para contar cada tipo individualmente.
# ============================================================

quarentenas = [
    ("clientes",         f"{SILVER}.quarentena_clientes"),
    ("contas",           f"{SILVER}.quarentena_contas"),
    ("cartoes",          f"{SILVER}.quarentena_cartoes"),
    ("transacoes_cartao", f"{SILVER}.quarentena_transacoes_cartao"),
    ("estornos",         f"{SILVER}.quarentena_estornos"),
]

# Constrói UNION ALL com EXPLODE/SPLIT para cada quarentena
unions = []
for entidade, t_quar in quarentenas:
    unions.append(f"""
    SELECT '{entidade}' AS entidade, tipo_erro, COUNT(*) AS qtd_registros
    FROM (
      SELECT EXPLODE(SPLIT(motivo_erro, ' [|] ')) AS tipo_erro
      FROM {t_quar}
      WHERE motivo_erro != ''
    ) exploded
    WHERE tipo_erro != ''
    GROUP BY tipo_erro
    """)

sql_alertas = " UNION ALL ".join(unions)

spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD}.gold_alertas_qualidade_particionado
USING DELTA
AS SELECT
  entidade,
  tipo_erro,
  SUM(qtd_registros)   AS qtd_registros,
  CURRENT_TIMESTAMP()  AS timestamp_processamento
FROM ({sql_alertas}) combined
GROUP BY entidade, tipo_erro
ORDER BY entidade, qtd_registros DESC
""")

qtd_aq = spark.table(f"{GOLD}.gold_alertas_qualidade_particionado").count()
print(f"GOLD_ALERTAS_QUALIDADE: {qtd_aq} combinações entidade x tipo_erro")
display(spark.sql(f"""
SELECT * FROM {GOLD}.gold_alertas_qualidade_particionado
ORDER BY entidade, qtd_registros DESC
"""))

In [0]:
# ============================================================
# 5. RESUMO DO PIPELINE GOLD PARTICIONADO + VALIDAÇÕES
# ============================================================
from pyspark.sql import functions as F

tabelas_gold = [
    ("gold_transacoes_cartao_particionado",        f"{GOLD}.gold_transacoes_cartao_particionado"),
    ("gold_resumo_cliente_mes_particionado",       f"{GOLD}.gold_resumo_cliente_mes_particionado"),
    ("gold_ranking_estabelecimento_particionado",   f"{GOLD}.gold_ranking_estabelecimento_particionado"),
    ("gold_alertas_qualidade_particionado",         f"{GOLD}.gold_alertas_qualidade_particionado"),
]

linhas = [(nome, spark.table(t).count()) for nome, t in tabelas_gold]
display(spark.createDataFrame(linhas, "tabela string, registros long"))

# --- Validações de consistência ---
gold_trans = spark.table(f"{GOLD}.gold_transacoes_cartao_particionado")
fact = spark.table(f"{GOLD}.fact_transacoes_financeiras")
silver_fin = spark.table(f"{SILVER}.transacoes_financeiras_particionado")

print("1. gold_transacoes == fact_transacoes_financeiras:",
      gold_trans.count(), "vs", fact.count())

soma_gold   = gold_trans.agg(F.sum("valor").alias("v")).collect()[0]["v"]
soma_fact   = fact.agg(F.sum("valor").alias("v")).collect()[0]["v"]
soma_silver = silver_fin.agg(F.sum("valor").alias("v")).collect()[0]["v"]
print("2. soma dos valores gold == fact:", float(soma_gold), "vs", float(soma_fact))
print("2b. rastreabilidade gold vs silver:", float(soma_gold), "vs", float(soma_silver))

soma_resumo = spark.table(f"{GOLD}.gold_resumo_cliente_mes_particionado") \
    .agg(F.sum("valor_total").alias("v")).collect()[0]["v"]
print("3. soma do resumo cliente-mês == soma gold:", float(soma_resumo), "vs", float(soma_gold))

soma_ranking = spark.table(f"{GOLD}.gold_ranking_estabelecimento_particionado") \
    .agg(F.sum("valor_total").alias("v")).collect()[0]["v"]
print("4. soma do ranking == soma gold:", float(soma_ranking), "vs", float(soma_gold))

total_rejeitados_silver = 0
for _, t in quarentenas:
    total_rejeitados_silver += spark.table(t).count()
total_alertas = spark.table(f"{GOLD}.gold_alertas_qualidade_particionado") \
    .agg(F.sum("qtd_registros")).collect()[0][0]
print("5. total alertas gold >= registros rejeitados na silver (motivos combinados contam em cada tipo):",
      total_alertas, "vs", total_rejeitados_silver)

# Partições por tabela Gold
print("\nPartições (ano_mes) por tabela Gold:")
for nome, t in tabelas_gold:
    try:
        partes = spark.sql(f"SHOW PARTITIONS {t}").count()
        print(f"  {nome}: {partes} partições")
    except:
        print(f"  {nome}: sem particionamento")

print("\nPipeline Gold particionado concluído com sucesso.")